# FCA–MoE v0.3: executable research scaffold

This notebook validates context construction, FCA enumeration, and a small-E pairwise-MaxEnt oracle. It does **not** claim pretrained-LLM, GPU, or KV-cache performance results.

**Repository:** `msellamiTN/FCA-MoE`, branch `research/fca-moe-v0.3`.

**Scientific boundary:** the scalable E=32 pairwise-MaxEnt sampler and 999-surrogate study are pending implementation gates. Synthetic routes are software-validation data only.

In [ ]:
# Clone the research branch and install its v0.3 dependencies
!git clone --depth 1 --branch research/fca-moe-v0.3 https://github.com/msellamiTN/FCA-MoE.git /content/FCA-MoE
!pip -q install -r /content/FCA-MoE/research/v03/requirements.txt
from pathlib import Path
import sys, numpy as np, pandas as pd
PROJECT_ROOT = Path('/content/FCA-MoE/research/v03')
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from moe_fca.synthetic import planted_routes
from moe_fca.contexts import token_context, window_context, chronological_split
from moe_fca.fca import next_closure, lattice_summary
from moe_fca.metrics import concept_count_profile
from moe_fca.maxent_exact import fit_pairwise_maxent, sample_exact
N_EXPERTS, TOP_K = 32, 2
routes = planted_routes(n_tokens=12000, n_experts=N_EXPERTS, top_k=TOP_K, seed=7)
T = token_context(routes, N_EXPERTS)
train, validation, test = chronological_split(T)
assert np.all(T.sum(axis=1) == TOP_K)
print('routes:', routes.shape, 'context:', T.shape)
print('chronological splits:', train.shape, validation.shape, test.shape)


## Window-context grid

The configured grid is windows 32/64/128 and thresholds 0.10/0.20/0.30.

In [ ]:
window_results = {}
for w in (32, 64, 128):
    for tau in (0.10, 0.20, 0.30):
        C = window_context(routes, w, tau, N_EXPERTS)
        window_results[(w, tau)] = C
        print((w, tau), C.shape, 'mean active experts=', round(float(C.sum(1).mean()), 3))


## FCA smoke test

We first project to eight attributes to keep unrestricted concept enumeration bounded. Full 32-attribute runs require a bounded/incremental FCA implementation and explicit concept-growth reporting.

In [ ]:
small = window_results[(128, 0.20)][:, :8]
intents = next_closure(small, max_concepts=100000)
print(lattice_summary(small, intents))
print(concept_count_profile(intents, small.shape[1]))


## Exact pairwise-MaxEnt oracle (small E only)

This enumerates all `2^E` binary states and is limited to E≤16. It validates moment matching; it is not the scalable E=32 sampler.

In [ ]:
rng = np.random.default_rng(42)
small_context = (rng.random((500, 8)) < np.array([.2,.25,.3,.35,.4,.45,.5,.55])).astype(np.uint8)
small_context[:, 1] = np.maximum(small_context[:, 1], small_context[:, 0] & (rng.random(500) < .5))
fit = fit_pairwise_maxent(small_context, max_experts=16)
surrogate = sample_exact(fit, 20000, seed=123)
print('optimizer:', fit['optimizer_success'], fit['optimizer_message'])
print('max moment error:', fit['moment_max_abs_error'])
print('sample shape:', surrogate.shape)


## Interpretation guardrails

- Pairwise moment preservation defines the chosen MaxEnt null; it does not prove that every higher-order dependency is absent.
- FCA concept counts are not proxies for routing quality, GPU utilization, KV-cache hit rate, latency, throughput, memory, communication, or energy.
- KV-cache claims require separate systems experiments with explicit cache policy, realistic sequence workloads, GPU measurements, and baselines.
- The scalable E=32 sampler must be validated against this exact oracle at small E before the 999-surrogate protocol is run.